
# Credit Card Fraud — Decision Trees & KNN Tournament (Synthetic Data)

Este cuaderno crea un dataset **sintético** de transacciones con tarjeta de crédito para clasificar si una transacción es **fraudulenta** o **no**.  
Incluye:

- Generación de datos sintéticos (con leve desbalance)
- **EDA** (análisis exploratorio de datos)
- **Feature Engineering** (One-Hot Encoding y escalamiento)
- **Cross-Validation**
- Modelos:
  - Árbol de Decisión (gini, parámetros por defecto)
  - Árbol de Decisión (gini, **max_depth** ajustado)
  - Árbol de Decisión (**entropy**, **max_depth** ajustado)
  - **KNN** con `weights='uniform'`
  - **KNN** con `weights='distance'`
- Reportes de clasificación, matrices de confusión y un **leaderboard** comparativo


In [ ]:

# Imports básicos
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, accuracy_score, f1_score, precision_score, recall_score

from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

# Reproducibilidad
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option('display.max_columns', None)


## 1) Generación de datos sintéticos

In [ ]:

# Parámetros del dataset
N = 20_000             # número de transacciones
fraud_rate = 0.035     # ~3.5% de fraudes

# Variables numéricas (simuladas)
amount = np.random.gamma(shape=2.0, scale=40.0, size=N)  # montos positivos, sesgados
time_since_last = np.random.exponential(scale=60.0, size=N)  # minutos desde la última compra
customer_age = np.random.normal(loc=36, scale=10, size=N).clip(18, 85)
merchant_risk_score = np.random.beta(a=2, b=5, size=N) * 100  # 0-100

# Variables categóricas
mcc = np.random.choice(['5411_grocery','5812_restaurant','5311_dept_store','4814_telco','5944_jewelry','4899_streaming'], size=N, p=[0.22,0.28,0.18,0.12,0.07,0.13])
channel = np.random.choice(['POS','ECOM','MOTO'], size=N, p=[0.58,0.36,0.06])
country = np.random.choice(['SV','US','MX','CO','PA','ES'], size=N, p=[0.35,0.25,0.15,0.1,0.1,0.05])
hour = np.random.randint(0, 24, size=N)

# Probabilidad base de fraude
base_logit = -3.2  # ~4%
# Efectos (heurísticos) que elevan el riesgo
logit = (
    base_logit
    + 0.015*(amount-50)                            # montos altos
    + 0.012*(merchant_risk_score-40)               # comercio más riesgoso
    + 0.01*np.where(channel=='ECOM', 1, 0)         # ecommerce
    + 0.02*np.where(channel=='MOTO', 1, 0)         # mail/phone order
    + 0.018*np.where((hour<5) | (hour>23), 1, 0)   # horas atípicas
    - 0.006*(customer_age-35)                      # usuarios mayores un poco menos riesgosos
    + 0.008*np.where(mcc=='5944_jewelry', 1, 0)    # rubros más sensibles
    + 0.006*np.where(np.isin(country, ['US','ES']), 1, 0)

)

# Ajuste para acercarnos al fraud_rate deseado
# Escalamos el logit para que la media de probas se aproxime al target
from scipy.special import expit

def calibrate_logit_to_rate(logit, target_rate, lr=0.01, iters=300):
    offset = 0.0
    for _ in range(iters):
        p = expit(logit + offset)
        diff = p.mean() - target_rate
        if abs(diff) < 1e-6:
            break
        offset -= lr * diff
    return logit + offset

logit_cal = calibrate_logit_to_rate(logit, fraud_rate)
proba = expit(logit_cal)

y = (np.random.rand(N) < proba).astype(int)

df = pd.DataFrame({
    'amount': amount,
    'time_since_last_min': time_since_last,
    'customer_age': customer_age,
    'merchant_risk_score': merchant_risk_score,
    'mcc': mcc,
    'channel': channel,
    'country': country,
    'hour': hour,
    'is_fraud': y
})

df.sample(8, random_state=RANDOM_STATE)


## 2) EDA (Exploratory Data Analysis)

In [ ]:

# Distribución de la clase
class_counts = df['is_fraud'].value_counts().sort_index()
print("Distribución de clases (0=no fraude, 1=fraude):")
print(class_counts, "\n")
print("Proporción de fraude:", round(df['is_fraud'].mean(), 4))

plt.figure()
plt.bar(class_counts.index.astype(str), class_counts.values)
plt.title("Distribución de la clase (is_fraud)")
plt.xlabel("Clase")
plt.ylabel("Frecuencia")
plt.show()


In [ ]:

# Histograma de montos
plt.figure()
plt.hist(df['amount'], bins=50)
plt.title("Histograma de montos (amount)")
plt.xlabel("amount")
plt.ylabel("Frecuencia")
plt.show()


In [ ]:

# Boxplot: amount por clase
plt.figure()
plt.boxplot([df.loc[df.is_fraud==0,'amount'], df.loc[df.is_fraud==1,'amount']], labels=['No fraude','Fraude'])
plt.title("Amount por clase")
plt.ylabel("amount")
plt.show()


In [ ]:

# Correlaciones (solo numéricas)
num_cols = ['amount','time_since_last_min','customer_age','merchant_risk_score','hour','is_fraud']
corr = df[num_cols].corr()
print(corr)

plt.figure()
plt.imshow(corr, interpolation='nearest')
plt.xticks(range(len(num_cols)), num_cols, rotation=45, ha='right')
plt.yticks(range(len(num_cols)), num_cols)
plt.title("Matriz de correlación (numéricas)")
plt.colorbar()
plt.tight_layout()
plt.show()


## 3) Feature Engineering y Split

In [ ]:

X = df.drop(columns=['is_fraud'])
y = df['is_fraud']

numeric_features = ['amount','time_since_last_min','customer_age','merchant_risk_score','hour']
categorical_features = ['mcc','channel','country']

preprocess = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

print(X_train.shape, X_test.shape, y_train.mean(), y_test.mean())


## 4) Cross-Validation Helper

In [ ]:
def evaluate_model(name, pipeline, X_train, y_train, X_test, y_test, cv_splits=5):
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=RANDOM_STATE)
    # Usamos ROC-AUC como métrica de CV (apropiada para clases desbalanceadas)
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='roc_auc')

    # Entrenamiento final y predicciones en test
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    try:
        y_proba = pipeline.predict_proba(X_test)[:, 1]
        roc = roc_auc_score(y_test, y_proba)
    except Exception:
        y_proba = None
        roc = np.nan

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)

    # Resumen numérico
    print(f"\n=== {name} ===")
    print("Cross-validated ROC-AUC (mean ± std): ", round(cv_scores.mean(), 4), "±", round(cv_scores.std(), 4))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, digits=4, zero_division=0))

    # 1) Matriz de confusión
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No fraude','Fraude'])
    plt.figure()
    disp.plot(values_format='d')
    plt.title(f"Matriz de confusión — {name}")
    plt.show()

    # 2) Distribución de ROC-AUC en CV (boxplot)
    plt.figure()
    plt.boxplot(cv_scores, vert=True)
    plt.title(f"Cross-Validation ROC-AUC — {name}")
    plt.ylabel("ROC-AUC")
    plt.show()

    # 3) Puntajes por fold (línea simple)
    plt.figure()
    plt.plot(range(1, len(cv_scores)+1), cv_scores, marker='o')
    plt.title(f"ROC-AUC por fold — {name}")
    plt.xlabel("Fold")
    plt.ylabel("ROC-AUC")
    plt.xticks(range(1, len(cv_scores)+1))
    plt.show()

    # 4) Curva ROC en test (si hay probabilidades)
    if y_proba is not None:
        from sklearn.metrics import RocCurveDisplay
        plt.figure()
        RocCurveDisplay.from_predictions(y_test, y_proba)
        plt.title(f"Curva ROC (test) — {name}")
        plt.show()

    return {
        'model': name,
        'cv_roc_auc_mean': cv_scores.mean(),
        'test_roc_auc': roc,
        'test_accuracy': acc,
        'test_f1': f1,
        'test_precision': prec,
        'test_recall': rec,
    }


## 5) Modelos

In [ ]:

models_summary = []

dt_default = Pipeline(steps=[
    ('prep', preprocess),
    ('model', DecisionTreeClassifier(random_state=RANDOM_STATE))
])

res_dt_default = evaluate_model("DecisionTree (gini, default)", dt_default, X_train, y_train, X_test, y_test)
models_summary.append(res_dt_default)


In [ ]:

dt_depth = Pipeline(steps=[
    ('prep', preprocess),
    ('model', DecisionTreeClassifier(random_state=RANDOM_STATE, max_depth=6))
])

res_dt_depth = evaluate_model("DecisionTree (gini, max_depth=6)", dt_depth, X_train, y_train, X_test, y_test)
models_summary.append(res_dt_depth)


In [ ]:

dt_entropy = Pipeline(steps=[
    ('prep', preprocess),
    ('model', DecisionTreeClassifier(random_state=RANDOM_STATE, criterion='entropy', max_depth=6))
])

res_dt_entropy = evaluate_model("DecisionTree (entropy, max_depth=6)", dt_entropy, X_train, y_train, X_test, y_test)
models_summary.append(res_dt_entropy)


In [ ]:

knn_uniform = Pipeline(steps=[
    ('prep', preprocess),
    ('model', KNeighborsClassifier(n_neighbors=5, weights='uniform'))
])

res_knn_uniform = evaluate_model("KNN (weights='uniform')", knn_uniform, X_train, y_train, X_test, y_test)
models_summary.append(res_knn_uniform)


In [ ]:

knn_distance = Pipeline(steps=[
    ('prep', preprocess),
    ('model', KNeighborsClassifier(n_neighbors=5, weights='distance'))
])

res_knn_distance = evaluate_model("KNN (weights='distance')", knn_distance, X_train, y_train, X_test, y_test)
models_summary.append(res_knn_distance)


## 6) Leaderboard de Modelos

In [ ]:

leaderboard = pd.DataFrame(models_summary).sort_values(by=['test_f1','test_roc_auc','test_accuracy'], ascending=False)
leaderboard.reset_index(drop=True, inplace=True)
leaderboard
